# BTS Digital Twin (NVS) — Vòng 3: error-guided refine (tinh chỉnh có hướng dẫn bởi lỗi)

**Đọc trước:** `docs/00_MASTER_PLAN.md` mục 3.2 + `docs/PORTED_KNOWLEDGE.md` mục 3 —
giải thích đầy đủ cơ chế, cạm bẫy đã tránh, và giả định kỹ thuật CHƯA có bằng chứng
thực nghiệm (lỗi đo trên ảnh train có tương quan với lỗi ở test/holdout hay không).

Notebook này nạp checkpoint **Vòng 2** (đã tải lên Drive), tự đo vùng
pixel còn lỗi cao (so với ảnh train GT thật), tinh chỉnh NGẮN ưu tiên vùng đó, rồi
**tự đo Score holdout TRƯỚC/SAU** để biết ngay có nên giữ kết quả Vòng 3 này
hay không — không suy đoán/tin trực giác (bài học từ repo tiền nhiệm: depth-prior/
antenna-focus đo thật KHÔNG cải thiện dù trực giác nghĩ sẽ cải thiện).

Yêu cầu: checkpoint Vòng 2 đã train ở `MODE="holdout"` (cần GT holdout
để tự đo Score) và **KHÔNG** dùng antenna-focus (không tương thích, xem
`apply_error_refine_patch.py`).

## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
# DỪNG NGAY nếu không có GPU — trước đây cell này CHỈ print() cảnh báo rồi để "Run All"
# chạy tiếp bình thường (không raise/assert gì) — dễ bị bỏ sót giữa hàng loạt dòng log
# của !pip install ngay cell sau, dẫn tới train/render "chạy" hàng chục phút/vài tiếng
# trên CPU rồi mới crash ở lần gọi .cuda() đầu tiên (torch.cuda.is_available()=False ->
# RuntimeError muộn, sau khi đã tốn thời gian tải dataset/build extension) — rất tốn
# thời gian dưới áp lực deadline (xem docs/00_MASTER_PLAN.md mục 1). Dừng NGAY tại đây
# với thông báo rõ ràng thay vì để lỗi lộ ra muộn và mù mờ hơn nhiều.
if not torch.cuda.is_available():
    raise SystemExit(
        "KHÔNG có GPU khả dụng (torch.cuda.is_available()=False). Vào Settings (góc phải) "
        "-> Accelerator -> chọn GPU T4 x2 hoặc P100 -> Save, rồi chạy lại từ đầu. "
        "KHÔNG chạy tiếp các cell sau khi chưa có GPU — train/render 3DGS cần CUDA, chạy "
        "trên CPU sẽ crash muộn (sau khi đã tốn thời gian tải dataset/cài đặt) hoặc treo "
        "vô thời hạn."
    )
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q pycolmap "scikit-image>=0.19" lpips plyfile tqdm gdown

## Bước 2 — Clone + build 3D Gaussian Splatting

Clone SẠCH (chưa vá gì) — patch error-refine áp ở Bước 6.

In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /kaggle/working/gaussian-splatting
!git checkout 54c035f7834b564019656c3e3fcc3646292f727d
!git submodule update --init --recursive
%cd /kaggle/working
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = "/kaggle/working/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])

## Bước 3 — Lấy code pipeline từ Git repo của bạn (khuyến nghị để **Private**)

Repo Private vẫn clone được bình thường trên Kaggle, chỉ cần xác thực bằng
**Personal Access Token (PAT)** thay vì mật khẩu. Các bước 1 lần:

1. Đẩy code lên GitHub, chọn **Private** khi tạo repo (không ai ngoài bạn xem được,
   kể cả khi bạn share notebook Kaggle này cho người khác sau này).
2. Tạo token: GitHub → **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Chọn:
   - Repository access: **Only select repositories** → chọn đúng repo vừa tạo.
   - Permissions → Contents: **Read-only** (không cần quyền gì khác).
   - Đặt ngày hết hạn (Expiration) ngắn thôi, vd 30-90 ngày — hết hạn thì tạo token mới.
3. **Copy token, dán vào Kaggle Secrets (KHÔNG dán thẳng vào code)**: trong notebook
   Kaggle, vào menu **Add-ons → Secrets → Add a new secret** → Label đặt đúng tên
   `GITHUB_TOKEN`, Value dán token vừa copy → Save. Cell bên dưới sẽ tự đọc secret
   này lúc chạy, token không hề xuất hiện trong code/notebook — kể cả nếu lỡ share
   notebook cho người khác, họ cũng không nhìn thấy được token của bạn.

Cell dò-thư-mục bên dưới tự tìm thư mục con tên `pipeline` (chứa `common/` và
`scripts/`) ở bất kỳ độ sâu nào trong repo vừa clone, không cần đúng ngay gốc repo.

In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin-MultiRound.git"
GIT_BRANCH = "main"  # <-- đổi nếu code Vòng 1 đang nằm ở nhánh khác chưa merge vào main

# GITHUB_TOKEN: ưu tiên lấy từ Kaggle Secrets (an toàn, không lộ trong code).
# Chỉ cần dán thẳng vào biến bên dưới nếu bạn KHÔNG dùng Kaggle Secrets (kém an
# toàn hơn — token sẽ nằm lộ trong notebook, đừng share notebook cho ai nếu làm vậy).
GITHUB_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
except Exception:
    if not GITHUB_TOKEN:
        print("Không tìm thấy Kaggle Secret 'GITHUB_TOKEN' (bỏ qua nếu repo Public, "
              "hoặc bạn đã dán token thẳng vào biến GITHUB_TOKEN ở trên).")

assert REPO_URL, "Chưa điền REPO_URL — dán link git repo chứa thư mục pipeline/ vào biến này rồi chạy lại cell."

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf /kaggle/working/_repo_clone
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/_repo_clone

In [ ]:
# Tự dò thư mục "pipeline" (chứa common/ và scripts/) ở bất kỳ đâu trong repo vừa
# clone, rồi symlink về /kaggle/working/pipeline — mọi cell sau đều gọi script từ đây.
import os
import shutil
from pathlib import Path

CLONE_ROOT = Path("/kaggle/working/_repo_clone")
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None and (CLONE_ROOT / "common").exists() and (CLONE_ROOT / "scripts").exists():
    found = CLONE_ROOT  # trường hợp bạn push thẳng NỘI DUNG pipeline/ làm gốc repo

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'pipeline' (chứa common/ và scripts/) trong repo vừa clone.\n"
        f"Nội dung clone nằm ở {CLONE_ROOT} — kiểm tra lại đã push đúng thư mục pipeline/ lên git chưa."
    )

print("Tìm thấy code pipeline tại:", found)
target = Path("/kaggle/working/pipeline")
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink -> /kaggle/working/pipeline ->", found.resolve())

Path("/kaggle/working/pipeline/work").mkdir(parents=True, exist_ok=True)

## Bước 4 — Tải dataset từ Google Drive

Điền link chia sẻ Google Drive (chế độ "Anyone with the link") vào `GDRIVE_URL` bên
dưới — file phải là **1 file .zip** chứa `Dataset/VAI_NVS_DATA_ROUND2/<scene>/...`
(zip nguyên thư mục `Dataset`, hoặc chỉ riêng `VAI_NVS_DATA_ROUND2` cũng được — cell
dưới tự dò tìm thư mục chứa các scene ở bất kỳ độ sâu nào trong zip, KHÔNG bắt buộc
đúng tên thư mục bọc ngoài, xem `docs/PORTED_KNOWLEDGE.md` mục 1).

In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/178EL7jCSVD59q19SMpeOgnOfOIC66I_t/view?usp=drive_link"

assert GDRIVE_URL, "Chưa điền GDRIVE_URL — dán link chia sẻ Google Drive (Anyone with the link) của file zip dataset vào biến này rồi chạy lại cell."

import os
os.makedirs("/kaggle/working/_dataset_raw", exist_ok=True)
!gdown --fuzzy "{GDRIVE_URL}" -O /kaggle/working/dataset.zip
!unzip -q -o /kaggle/working/dataset.zip -d /kaggle/working/_dataset_raw
print("Đã giải nén xong, đang dò tìm thư mục chứa các scene ...")

In [ ]:
# Tự dò thư mục chứa các scene phẳng (HCM0421/, chair/, bonsai/...) ở bất kỳ đâu
# trong zip vừa giải nén, rồi symlink về đúng vị trí mà pipeline/common/scenes.py
# cần: /kaggle/working/Dataset/VAI_NVS_DATA_ROUND2
#
# KHÔNG bắt buộc thư mục bọc ngoài phải tên đúng "VAI_NVS_DATA_ROUND2" — chỉ cần
# TÌM ĐƯỢC 1 thư mục (kể cả chính gốc giải nén, nếu zip không có lớp bọc ngoài)
# chứa đủ NHIỀU scene mong đợi trực tiếp bên trong. Logic này đã được sửa (fix)
# thành linh hoạt ở repo tiền nhiệm — bản cũ bắt buộc đúng tên thư mục nên sẽ báo
# lỗi "Không tìm thấy..." nếu file zip giải nén ra không có đúng lớp thư mục tên
# "VAI_NVS_DATA_ROUND2" đó (vd giải nén thẳng ra HCM0421/ ở gốc, hoặc thư mục bọc
# ngoài đặt tên khác) — dù dữ liệu vẫn đầy đủ. Đây là bản đã fix, port nguyên vẹn.
#
# Danh sách tên scene lặp lại thủ công ở đây (không import common.scenes) vì
# sys.path chưa trỏ tới pipeline/ ở bước này (việc đó làm ở cell kiểm tra ngay
# sau) — giữ đồng bộ với BTS_SCENES/GENERIC_SCENES trong pipeline/common/scenes.py
# nếu sau này thêm/bớt scene.
import os
from pathlib import Path

_expected_scene_dirs = {"HCM0421", "HCM0539", "HCM0540", "HCM0644", "HCM0674", "bonsai", "chair"}
_MIN_MATCH = 4  # đủ scene trùng khớp để tin đây đúng là thư mục dataset (tránh khớp nhầm thư mục rác)

RAW_ROOT = Path("/kaggle/working/_dataset_raw")
found = None
best_match = 0
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    # __MACOSX/ là rác do Mac tạo khi nén zip — nó TỰ NHÂN BẢN y hệt cấu trúc thư
    # mục thật (HCM0421/, train/images/...) nhưng file bên trong chỉ là file
    # rác metadata "._<tên file>", không phải dữ liệu thật. Phải loại trừ, nếu
    # không os.walk có thể tìm trúng "__MACOSX/..." trước bản thật.
    dirnames[:] = [d for d in dirnames if d != "__MACOSX" and not d.startswith(".")]
    n_match = len(_expected_scene_dirs & set(dirnames))
    if n_match > best_match:
        best_match = n_match
        found = Path(dirpath)
    if n_match == len(_expected_scene_dirs):
        break  # khớp đủ cả 7 — dừng sớm, khỏi walk tiếp cho nhanh

if found is None or best_match < _MIN_MATCH:
    raise SystemExit(
        f"Không tìm thấy thư mục nào chứa >= {_MIN_MATCH}/{len(_expected_scene_dirs)} scene mong đợi "
        f"bên trong {RAW_ROOT}. Khớp tốt nhất: {found} ({best_match} scene). "
        f"Kiểm tra lại file zip GDRIVE_URL có đúng dataset không."
    )

print(f"Tìm thấy thư mục dataset tại: {found} ({best_match}/{len(_expected_scene_dirs)} scene khớp)")

target_parent = Path("/kaggle/working/Dataset")
target_parent.mkdir(parents=True, exist_ok=True)
target = target_parent / "VAI_NVS_DATA_ROUND2"
if target.is_symlink() or target.exists():
    if target.is_symlink():
        target.unlink()
    else:
        import shutil
        shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink ->", target, "->", found.resolve())

os.environ["BTS_DATASET_ROOT"] = str(target)
print("BTS_DATASET_ROOT =", os.environ["BTS_DATASET_ROOT"])

In [ ]:
# Kiểm tra lại: liệt kê đủ 7 scene + scene nào có sparse hợp lệ — dataset đầy đủ
# thì kỳ vọng has_valid_provided_sparse=True cho CẢ 7 scene.
# Nếu train_ok=False hết cho mọi scene, kiểm tra lại BTS_DATASET_ROOT ở cell trên
# có trỏ đúng chỗ chứa VAI_NVS_DATA_ROUND2 hay không (thường do dataset.zip
# tải/giải nén thiếu — thử xoá /kaggle/working/_dataset_raw và tải lại từ đầu).
import sys
sys.path.insert(0, "/kaggle/working/pipeline")
from common.scenes import all_scenes, DATASET_ROOT

print("DATASET_ROOT =", DATASET_ROOT, "| tồn tại:", DATASET_ROOT.exists())
for s in all_scenes():
    ok_train = s.train_images_dir.exists()
    ok_csv = s.test_poses_csv.exists()
    n_train = len(list(s.train_images_dir.glob("*"))) if ok_train else 0
    print(f"{s.name:10s} {s.domain:8s} train_ok={ok_train} n_train={n_train:4d} "
          f"csv_ok={ok_csv} has_valid_provided_sparse={s.has_valid_provided_sparse()}")

## Bước 5 — Cấu hình scene + checkpoint Vòng 2 có sẵn

In [ ]:
SCENE = "HCM0421"
CHECKPOINT_DRIVE_LINK = ""  # <-- dán link Drive tới thư mục gs_model của checkpoint Vòng 2 (không phải file .ply đơn)
REFINE_ITERATIONS = 3000    # <-- ngân sách tinh chỉnh THÊM của vòng này (không phải train lại từ đầu)
MAX_ERROR_WEIGHT = 6.0      # <-- trọng số loss tối đa ở vùng lỗi cao nhất

assert CHECKPOINT_DRIVE_LINK, "Chưa điền CHECKPOINT_DRIVE_LINK — dán link Drive tới thư mục gs_model của checkpoint Vòng 2."
print(f"SCENE={SCENE}  REFINE_ITERATIONS={REFINE_ITERATIONS}  MAX_ERROR_WEIGHT={MAX_ERROR_WEIGHT}")

## Bước 6 — Tải checkpoint Vòng 2 từ Google Drive

In [ ]:
import shutil
from pathlib import Path

dest_dir = Path(f"/kaggle/working/pipeline/work/{SCENE}")
dest_dir.mkdir(parents=True, exist_ok=True)
gs_model_dst = dest_dir / "gs_model"
shutil.rmtree(gs_model_dst, ignore_errors=True)

raw_dl_dir = Path(f"/kaggle/working/_ckpt_raw/{SCENE}")
shutil.rmtree(raw_dl_dir, ignore_errors=True)
raw_dl_dir.mkdir(parents=True, exist_ok=True)
print(f"===== {SCENE}: tải thư mục gs_model từ Drive =====")
!gdown --fuzzy --folder "{CHECKPOINT_DRIVE_LINK}" -O "{raw_dl_dir}"

# gdown --folder có thể tự thêm 1 lớp thư mục con — tự dò lớp chứa "cfg_args" (file
# luôn nằm ở gốc gs_model/, do train.py tự ghi), không giả định cứng độ sâu.
candidates = [p.parent for p in raw_dl_dir.rglob("cfg_args")]
assert candidates, (
    f"Tải xong nhưng KHÔNG tìm thấy file 'cfg_args' trong {raw_dl_dir} — kiểm tra lại "
    f"link Drive có đúng là THƯ MỤC gs_model/ (không phải chỉ mỗi point_cloud.ply) và đã "
    f"share \"Anyone with the link\" chưa.")
src_root = candidates[0]
assert (src_root / "pipeline_train_flags.json").exists(), (
    f"Thiếu pipeline_train_flags.json trong {src_root} — thư mục gs_model tải lên Drive "
    f"phải nguyên vẹn (không tự xoá bớt file con nào).")

import json as _json
_flags = _json.loads((src_root / "pipeline_train_flags.json").read_text())
assert not _flags.get("antenna_focus", False), (
    "Checkpoint này train với ANTENNA_FOCUS=1 — KHÔNG tương thích với error-refine.")

shutil.copytree(src_root, gs_model_dst)
shutil.rmtree(raw_dl_dir, ignore_errors=True)

CKPT_ITERATION = max(int(p.name.split("_")[-1]) for p in (gs_model_dst / "point_cloud").glob("iteration_*"))
print(f"-> OK, {gs_model_dst} — checkpoint nguồn ở iteration {CKPT_ITERATION}")
print(f"-> pipeline_train_flags.json: {_flags}")

## Bước 7 — Tái tạo `colmap/dense/{images/,sparse/0/}`

Bước train trước đã tự dọn `dense/images/` sau khi xong (dọn đĩa bình thường) — mask
lỗi cần ảnh ĐÃ undistort chính xác pixel-for-pixel, không dùng bản xấp xỉ. Deterministic
(seed=42 cố định) nên tái tạo lại đúng holdout split/undistort như checkpoint gốc.

In [ ]:
holdout_dir = f"/kaggle/working/pipeline/work/{SCENE}/holdout"
import os
if not os.path.isdir(holdout_dir):
    !python /kaggle/working/pipeline/scripts/00_make_holdout_split.py --scene {SCENE}
else:
    print(f"Đã có {holdout_dir} — bỏ qua tạo lại.")
!python /kaggle/working/pipeline/scripts/01_run_colmap.py --scene {SCENE} --holdout

## Bước 8 — Đo Score TRƯỚC khi tinh chỉnh Vòng 3 (baseline so sánh)

In [ ]:
!python /kaggle/working/pipeline/scripts/03_render_test_poses.py --scene {SCENE} \
    --poses_csv /kaggle/working/pipeline/work/{SCENE}/holdout/holdout_poses.csv \
    --out_dir /kaggle/working/pipeline/work/{SCENE}/holdout_renders \
    --iteration {CKPT_ITERATION}
!python /kaggle/working/pipeline/scripts/04_eval_metrics.py --scene {SCENE}

import shutil
shutil.copy(
    f"/kaggle/working/pipeline/work/{SCENE}/eval_metrics.csv",
    f"/kaggle/working/pipeline/work/{SCENE}/eval_metrics_BEFORE_round3.csv",
)
print("-> Đã lưu bản sao eval_metrics_BEFORE_round3.csv để so sánh sau Bước 11.")

## Bước 9 — Sinh error mask (render lại pose TRAIN, so GT thật)

In [ ]:
!python /kaggle/working/pipeline/scripts/05_generate_error_mask.py --scene {SCENE} \
    --iteration {CKPT_ITERATION} --max_weight {MAX_ERROR_WEIGHT}

## Bước 10 — Vá `GS_REPO` + tinh chỉnh Vòng 3

In [ ]:
!python /kaggle/working/pipeline/scripts/apply_error_refine_patch.py --gs_repo {os.environ['GS_REPO']}

import os as _os
_os.environ["REFINE_ITERATIONS"] = str(REFINE_ITERATIONS)
_os.environ["MAX_ERROR_WEIGHT"] = str(MAX_ERROR_WEIGHT)
_os.environ["CKPT_ITERATION"] = str(CKPT_ITERATION)

!bash /kaggle/working/pipeline/scripts/06_train_refine.sh {SCENE}

## Bước 11 — Đo Score SAU Vòng 3, so với Bước 8

Nếu Score KHÔNG tăng (hoặc giảm), Vòng 3 KHÔNG có lợi cho scene này — đừng
dùng checkpoint đã refine, giữ checkpoint Vòng 2 gốc, coi như dừng ở
Vòng 2 cho scene này (không bắt buộc mọi scene đi hết N vòng).

In [ ]:
TARGET_ITERATION = CKPT_ITERATION + REFINE_ITERATIONS

!python /kaggle/working/pipeline/scripts/03_render_test_poses.py --scene {SCENE} \
    --poses_csv /kaggle/working/pipeline/work/{SCENE}/holdout/holdout_poses.csv \
    --out_dir /kaggle/working/pipeline/work/{SCENE}/holdout_renders \
    --iteration {TARGET_ITERATION}
!python /kaggle/working/pipeline/scripts/04_eval_metrics.py --scene {SCENE}

import csv

def _score_mean(path):
    rows = list(csv.DictReader(open(path)))
    return sum(float(r["score"]) for r in rows) / len(rows)

before = _score_mean(f"/kaggle/working/pipeline/work/{SCENE}/eval_metrics_BEFORE_round3.csv")
after = _score_mean(f"/kaggle/working/pipeline/work/{SCENE}/eval_metrics.csv")
print(f"Score TRƯỚC Vòng 3 (= Vòng 2): {before:.4f}")
print(f"Score SAU Vòng 3               : {after:.4f}")
print(f"Chênh lệch                        : {after - before:+.4f}")
if after > before:
    print(f"-> CÓ CẢI THIỆN — dùng checkpoint Vòng 3 (iteration {TARGET_ITERATION}) cho bước tiếp theo.")
else:
    print("-> KHÔNG cải thiện (hoặc tệ hơn) — GIỮ checkpoint Vòng 2, DỪNG LẠI ở scene này, "
          "không cần tải checkpoint Vòng 3 lên Drive.")

## Bước 12 (chỉ nếu Bước 11 cho thấy cải thiện) — Lưu checkpoint Vòng 3 lên Drive

Giống hệt "Bước 6" của `kaggle_round1_baseline.ipynb` — tải NGUYÊN thư mục
`pipeline/work/<SCENE>/gs_model/` (giờ có thêm `point_cloud/iteration_{TARGET_ITERATION}/`
+ `pipeline_train_flags.json` đã ghi thêm `refine_history`) lên Google Drive.

In [ ]:
print(f"Checkpoint Vòng 3 (nếu quyết định dùng): "
      f"pipeline/work/{SCENE}/gs_model/point_cloud/iteration_{TARGET_ITERATION}/point_cloud.ply")
print("Bấm Save Version, vào tab Output, tải nguyên thư mục gs_model/ về rồi upload lên "
      "Drive (đặt tên rõ <SCENE>_round3_gs_model) — cùng cách làm Bước 6 của "
      "kaggle_round1_baseline.ipynb.")